# Lecture: VAE on Real Images — Generating Faces with CelebA

So far the VAE was demonstrated on MNIST and Fashion-MNIST: small, grayscale,
28x28 images. Those datasets make the *mechanics* easy to see (especially the
2D latent space), but they hide how powerful a VAE becomes on **real, natural
images**.

This notebook applies the **same VAE principle** to [**CelebA**](https://mmlab.ie.cuhk.edu.hk/projects/CelebA.html),
a dataset of ~200,000 celebrity face photographs. We move to **RGB 64x64**
images and a much larger latent space (`latent_dim=256`).

The goal here is **not** to train from scratch — that needs a GPU and ~1-1.5 h
on a free Colab T4. Instead, this notebook is **load-only**: we load a
pre-trained model and explore what a face VAE can actually do:

1. **Reconstruction** — compress a face to 256 numbers and decode it back.
2. **Sampling from the prior** — invent entirely new faces of people who do not exist.
3. **Latent interpolation** — morph smoothly from one face into another.
4. **Attribute arithmetic** — find the "smile" or "glasses" direction in latent
   space and add it to any face.

To reproduce the weights yourself, the standalone training script
`train_vae_celeba.py` is provided for running on a GPU server (see the
final section).

## The Training Objective: Three Variants

All three shipped models share the **same architecture** (RGB 64x64 encoder /
decoder, `latent_dim=256`). They differ only in the **loss used during
training**, and that difference is exactly what drives image quality. The base
objective of every VAE is the negative ELBO:

$$\mathcal{L} \;=\; \underbrace{\mathbb{E}_{q_\psi(z|x)}\big[-\log p_\theta(x|z)\big]}_{\text{reconstruction}} \;+\; \beta \cdot \underbrace{\mathrm{KL}\big(q_\psi(z|x)\,\|\,\mathcal{N}(0,I)\big)}_{\text{regulariser}}$$

**1. Vanilla VAE — pixel likelihood, $\beta = 1$.**
The reconstruction term is a per-pixel loss (here binary cross-entropy). Because
the true posterior over images is multi-modal, minimising a per-pixel loss makes
the decoder output the **pixel-wise average** of all plausible reconstructions —
which looks **blurry**. This is the fundamental limitation of VAEs and the
motivation for GANs (Chapter C3).

**2. $\beta$-VAE with $\beta = 0.5$.**
Lowering $\beta$ down-weights the KL regulariser, so the model "spends" more
capacity on reconstruction fidelity and less on matching the prior. Faces get
somewhat sharper, but sampling from the prior degrades (the aggregated posterior
drifts further from $\mathcal{N}(0,I)$). It is a **fidelity-vs-prior trade-off**,
not a free lunch — and per-pixel blur remains.

**3. Perceptual VAE — VGG feature loss.**
The key change: instead of comparing $x$ and $\hat{x}$ pixel-by-pixel, compare
their **feature activations** in a frozen, pre-trained VGG16 network:

$$\mathcal{L}_{\text{recon}} \;=\; \|x-\hat{x}\|_2^2 \;+\; \lambda \sum_{l} \big\|\phi_l(x) - \phi_l(\hat{x})\big\|_2^2$$

where $\phi_l$ are VGG activations at selected layers. Feature space rewards
matching **edges, textures and structure** rather than exact pixel values, so the
decoder is no longer pushed towards the blurry mean. This gives the **sharpest**
faces of the three — though, being a VAE at 64x64, still not GAN-level crisp.

The comparison at the end of the notebook shows all three side by side.

## Setup (Google Colab only)

Run the following cell **only on Google Colab** to clone the repository and
copy the required `VAE_CelebA.py` module into the working directory. If you
work locally, skip it.

In [ ]:
!git clone https://github.com/Fjoelsak/AIBIP.git
!cp AIBIP/C2-Autoencoders/VAE_CelebA.py ./

## Loading the Pre-trained Model

The pre-trained weights (`vae_celeba.pth`, latent_dim=256) are committed to the
repository under `C2-Autoencoders/models/`. The cell below picks the correct
path automatically:

- **locally**: `models/vae_celeba.pth` (the notebook runs from this folder), or
- **on Colab**: `AIBIP/C2-Autoencoders/models/vae_celeba.pth` (after the clone
  in the setup cell).

No download is required.

In [ ]:
import os
import torch
from VAE_CelebA import VAE

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

LATENT_DIM = 256

# Three checkpoints are shipped, trained with increasingly better objectives.
# The main demos below use the best one; the final section compares all three.
def resolve(name):
    """Return the first existing path for a shipped file (local or Colab)."""
    for base in ("", "AIBIP/C2-Autoencoders/"):
        p = base + name
        if os.path.exists(p):
            return p
    return name

MODELS = {
    "vanilla (25ep, beta=1.0)":    "models/vae_celeba_vanilla.pth",
    "beta=0.5 (40ep)":             "models/vae_celeba_beta05.pth",
    "perceptual (VGG loss)":       "models/vae_celeba_perceptual.pth",
}
BEST = "perceptual (VGG loss)"

model = VAE(latent_dim=LATENT_DIM).to(device)
model.load_model(path=resolve(MODELS[BEST]), device=device)
model.eval()

## A Few Real Faces to Work With

For the demos we need a handful of real CelebA faces. To keep the notebook
runnable on Colab **without downloading the full CelebA dataset**, a small set
of pre-processed example faces is shipped with the repository in
`celeba_demo_assets/sample_faces.pt` (already center-cropped and resized to 64x64). They were
exported with `export_celeba_demo_assets.py` (see the final section).

In [ ]:
import os

# Pre-processed example faces: a dict with "images" (N, 3, 64, 64) in [0, 1].
CANDIDATE_PATHS = [
    "celeba_demo_assets/sample_faces.pt",                           # local run
    "AIBIP/C2-Autoencoders/celeba_demo_assets/sample_faces.pt",     # Colab after git clone
]
faces_path = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), CANDIDATE_PATHS[0])

faces = torch.load(faces_path, map_location=device)
images = faces["images"].to(device)
print("loaded", images.size(0), "example faces:", tuple(images.shape))

## 1) Reconstruction

The VAE compresses each 64x64x3 = 12,288-dimensional image into a 256-number
latent code and decodes it back. Reconstructions are recognisably the same
person but slightly smoothed — the hallmark "blurriness" of VAEs, caused by the
Gaussian likelihood and the averaging effect of the KL regulariser.

In [ ]:
import matplotlib.pyplot as plt

with torch.no_grad():
    recon, _, _ = model(images)

n = 8
fig, axes = plt.subplots(2, n, figsize=(2 * n, 4))
for i in range(n):
    axes[0, i].imshow(images[i].permute(1, 2, 0).cpu().numpy())
    axes[0, i].axis("off")
    axes[1, i].imshow(recon[i].permute(1, 2, 0).cpu().numpy())
    axes[1, i].axis("off")
axes[0, 0].set_ylabel("original",       rotation=0, ha="right", labelpad=40)
axes[1, 0].set_ylabel("reconstruction", rotation=0, ha="right", labelpad=40)
plt.tight_layout()
plt.show()

## 2) Sampling New Faces from the Prior

This is the capability a plain autoencoder lacks. Because the latent space is
regularised towards `N(0, I)`, we can simply draw random vectors `z ~ N(0, I)`
and decode them into **new faces of people who do not exist**.

In [ ]:
with torch.no_grad():
    samples = model.sample(16, device=device)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(samples[i].permute(1, 2, 0).cpu().numpy())
    ax.axis("off")
plt.suptitle("Faces sampled from the prior  z ~ N(0, I)")
plt.tight_layout()
plt.show()

## 3) Latent-Space Interpolation (Face Morphing)

Encoding two faces to their posterior means and decoding the straight line
between them produces a **smooth morph** — every intermediate point is a valid
face. This demonstrates that the latent space is continuous and semantically
meaningful, not just a lookup table of memorised images.

In [ ]:
steps = 10
morph = model.interpolate(images[0], images[1], steps=steps)

fig, axes = plt.subplots(1, steps, figsize=(2 * steps, 2.2))
for i in range(steps):
    axes[i].imshow(morph[i].permute(1, 2, 0).cpu().numpy())
    axes[i].axis("off")
plt.suptitle("Interpolation from face A (left) to face B (right)")
plt.tight_layout()
plt.show()

## 4) Attribute Arithmetic

CelebA ships 40 binary attribute labels per image (e.g. *Smiling*, *Eyeglasses*,
*Male*, *Blond_Hair*). The latent **direction** of an attribute is the
difference between the *mean latent code of images that have it* and the *mean
latent code of images that lack it*:

$$\mathbf{v}_{\text{attr}} = \overline{\mathbf{z}}_{\text{has attr}} - \overline{\mathbf{z}}_{\text{no attr}}$$

Estimating these directions requires encoding thousands of *labelled* images,
so they are **pre-computed** (with `export_celeba_demo_assets.py`) and shipped
in `celeba_demo_assets/attribute_vectors.pt`. Adding $\alpha \cdot \mathbf{v}_{\text{attr}}$ to a
face's latent code and decoding turns the attribute up or down continuously.

In [ ]:
# Pre-computed latent attribute directions for the best model.
ATTR_VECTORS = {
    "vanilla (25ep, beta=1.0)": "celeba_demo_assets/attribute_vectors_vanilla.pt",
    "beta=0.5 (40ep)":          "celeba_demo_assets/attribute_vectors_beta05.pt",
    "perceptual (VGG loss)":    "celeba_demo_assets/attribute_vectors_perceptual.pt",
}
vecs = torch.load(resolve(ATTR_VECTORS[BEST]), map_location=device)
attribute_vectors = {k: v.to(device) for k, v in vecs.items()}
print("available attribute directions:", list(attribute_vectors.keys()))

smile_vec = attribute_vectors["Smiling"]
print("'Smiling' direction, norm =", round(smile_vec.norm().item(), 3))

In [ ]:
# Apply the attribute vector to one face across a range of strengths.
face = images[5:6]
with torch.no_grad():
    z = model.encode(face)
    alphas = torch.linspace(-3, 3, 7, device=device)
    edited = torch.cat([model.decode(z + a * smile_vec) for a in alphas])

fig, axes = plt.subplots(1, len(alphas), figsize=(2 * len(alphas), 2.4))
for i, a in enumerate(alphas):
    axes[i].imshow(edited[i].permute(1, 2, 0).cpu().numpy())
    axes[i].set_title(f"a={a:.0f}")
    axes[i].axis("off")
plt.suptitle("Adding the 'Smiling' direction (left = less, right = more)")
plt.tight_layout()
plt.show()

## 5) Comparing the Three Training Objectives

Recall the three objectives introduced at the top of the notebook:

| Model | Objective | Epochs |
|-------|-----------|--------|
| **vanilla**    | pixel BCE, $\beta=1.0$            | 25 |
| **beta=0.5**   | pixel BCE, $\beta=0.5$            | 40 |
| **perceptual** | pixel MSE + VGG feature loss     | 40 |

The cell below shows reconstructions and prior samples for all three using the
**same** input faces and the **same** prior samples `z`, so the only variable is
the training loss. Watch how sharpness improves from top to bottom.

---
## Try It Yourself — Experiment with Faces

Work in pairs. **Predict first, then run, then explain in one sentence.**

**A. Which attribute is cleanest?** Re-run the attribute-arithmetic cell with
`Eyeglasses`, `Male`, and `Blond_Hair` instead of `Smiling`. Which attribute
direction works most reliably? Form a hypothesis *why* (hint: look at the
vector norms printed when loading the attribute vectors).

**B. Combine two attributes.** Add *two* attribute vectors at once (e.g.
`Smiling + Eyeglasses`) before decoding. Does it still work? What happens at
large strengths $\alpha$?

**C. Blind comparison.** Look at the three-model comparison figure *without*
the row labels. Can you tell which row is vanilla / beta / perceptual? What
visual cue gives it away?

**D. The honest limit.** Even the perceptual model is not as sharp as a GAN
would be. Describe the *specific* visual defect that remains, and argue why a
VAE cannot fully remove it. (This motivates GANs in Chapter C3.)

In [ ]:
# Load all three models and compare reconstruction + prior samples.
fixed_z = torch.randn(8, LATENT_DIM, device=device)   # same prior z for all models

cmp_models = {}
for name, path in MODELS.items():
    p = resolve(path)
    if not os.path.exists(p):
        print(f"skipping '{name}': {p} not found")
        continue
    m = VAE(latent_dim=LATENT_DIM).to(device)
    m.load_model(path=p, device=device)
    m.eval()
    cmp_models[name] = m

n = 8
rows = 1 + 2 * len(cmp_models)        # originals + (recon, prior) per model
fig, axes = plt.subplots(rows, n, figsize=(2 * n, 2 * rows))

for j in range(n):
    axes[0, j].imshow(images[j].permute(1, 2, 0).cpu().numpy())
    axes[0, j].axis("off")
axes[0, 0].set_ylabel("original", rotation=0, ha="right", labelpad=45)

with torch.no_grad():
    for r, (name, m) in enumerate(cmp_models.items()):
        recon, _, _ = m(images[:n])
        prior = m.decode(fixed_z)
        for j in range(n):
            axes[1 + 2 * r, j].imshow(recon[j].permute(1, 2, 0).cpu().numpy())
            axes[1 + 2 * r, j].axis("off")
            axes[2 + 2 * r, j].imshow(prior[j].permute(1, 2, 0).cpu().numpy())
            axes[2 + 2 * r, j].axis("off")
        axes[1 + 2 * r, 0].set_ylabel(f"{name} recon", rotation=0, ha="right", labelpad=45)
        axes[2 + 2 * r, 0].set_ylabel(f"{name} prior", rotation=0, ha="right", labelpad=45)

plt.tight_layout()
plt.show()

---
## Reproducing the Weights

This notebook is **load-only**. The three shipped checkpoints were produced with
the standalone script [`train_vae_celeba.py`](train_vae_celeba.py) on a GPU
server (training on CPU is impractical). CelebA images were read from a local
folder to avoid the rate-limited Google Drive mirror (e.g. the aligned images
from the Kaggle CelebA mirror):

```bash
# vanilla: pixel BCE, beta=1.0, 25 epochs
python train_vae_celeba.py --data-dir <imgs> --loss bce --beta 1.0     --epochs 25 --out models/vae_celeba_vanilla.pth

# beta=0.5: pixel BCE, lower KL weight, 40 epochs
python train_vae_celeba.py --data-dir <imgs> --loss bce --beta 0.5     --epochs 40 --out models/vae_celeba_beta05.pth

# perceptual: pixel MSE + VGG feature loss, 40 epochs
python train_vae_celeba.py --data-dir <imgs> --loss perceptual --beta 1.0     --epochs 40 --out models/vae_celeba_perceptual.pth
```

For each checkpoint, the matching attribute vectors are exported with
[`export_celeba_demo_assets.py`](export_celeba_demo_assets.py) via its
`--weights` and `--out-vectors` options. See each script's `--help` for all
options.